# EA1 - Actividad 1.4: Arquitecturas para Big Data

## Objetivos
- Comprender el ecosistema Hadoop y su evolucion
- Conocer las arquitecturas Lambda y Kappa
- Comparar soluciones on-premise vs cloud
- Identificar servicios equivalentes en GCP, AWS y Azure

## Conceptos Clave

Este notebook es **mayoritariamente conceptual**. La comprension de las arquitecturas de Big Data es fundamental para disenar sistemas escalables y eficientes.

Las preguntas clave que abordaremos:
- Como almacenamos y procesamos grandes volumenes de datos?
- Como manejamos datos en batch y en tiempo real?
- Cuando elegir on-premise vs cloud?

---
## 1. Ecosistema Hadoop

Hadoop fue el primer framework open-source para procesamiento distribuido de Big Data. Sus componentes principales son:

```
+================================================================+
|                    ECOSISTEMA HADOOP                            |
+================================================================+
|                                                                |
|  +----------+  +----------+  +----------+  +----------+       |
|  |   Hive   |  |   Pig    |  |  Sqoop   |  |  Flume   |       |
|  | (SQL)    |  | (Script) |  | (Import) |  | (Ingest) |       |
|  +----+-----+  +----+-----+  +----+-----+  +----+-----+       |
|       |              |              |              |            |
|  +----v--------------v--------------v--------------v-----+     |
|  |                  MapReduce                            |     |
|  |            (Motor de Procesamiento)                   |     |
|  +------------------------+------------------------------+     |
|                           |                                    |
|  +------------------------v------------------------------+     |
|  |                     YARN                              |     |
|  |          (Gestor de Recursos del Cluster)             |     |
|  +------------------------+------------------------------+     |
|                           |                                    |
|  +------------------------v------------------------------+     |
|  |                     HDFS                              |     |
|  |       (Sistema de Archivos Distribuido)               |     |
|  +-------------------------------------------------------+     |
+================================================================+
```

| Componente | Funcion |
|------------|----------|
| **HDFS** | Sistema de archivos distribuido. Almacena datos en bloques de 128MB replicados en multiples nodos |
| **YARN** | Gestor de recursos. Asigna CPU y memoria a las aplicaciones del cluster |
| **MapReduce** | Modelo de procesamiento: Map (transformar) + Reduce (agregar). Escribe resultados intermedios a disco |
| **Hive** | SQL sobre Hadoop. Traduce consultas SQL a jobs MapReduce |
| **Pig** | Lenguaje de scripting para flujos de datos |
| **Sqoop** | Importar/exportar datos entre HDFS y bases de datos relacionales |
| **Flume** | Ingesta de datos en streaming hacia HDFS |

## 2. Ecosistema Moderno

El ecosistema ha evolucionado significativamente. Hoy en dia, muchas herramientas complementan o reemplazan componentes originales de Hadoop:

```
+-----------------------------------------------------------------+
|                  ECOSISTEMA MODERNO                              |
+-----------------------------------------------------------------+
|                                                                 |
|  Ingesta          Procesamiento       Almacenamiento   Consulta |
|  +--------+       +------------+      +------------+  +------+ |
|  | Kafka  | ----> |   Spark    | ---> |  HDFS /    |  | Hive | |
|  | NiFi   |       |  (Batch +  |      |  S3 /      |  |Presto| |
|  | Flume  |       |  Streaming)|      |  Cloud     |  |Trino | |
|  +--------+       +------------+      | Storage    |  +------+ |
|                                       +------------+           |
|                                                                 |
|  Orquestacion: Airflow, Oozie, Prefect                         |
|  Coordinacion: ZooKeeper                                        |
|  Formato de datos: Parquet, ORC, Avro, Delta Lake               |
+-----------------------------------------------------------------+
```

| Herramienta | Rol | Ventaja sobre Hadoop clasico |
|-------------|-----|------------------------------|
| **Apache Spark** | Procesamiento batch y streaming | 10-100x mas rapido que MapReduce (procesamiento en memoria) |
| **Apache Kafka** | Plataforma de streaming de eventos | Baja latencia, alta capacidad, persistencia de mensajes |
| **Apache Hive** | Data warehouse sobre Hadoop | SQL familiar, optimizado con Tez |
| **Presto/Trino** | Motor de consultas SQL | Consultas interactivas sobre multiples fuentes de datos |
| **Apache Airflow** | Orquestacion de pipelines | DAGs programaticos en Python, monitoreo visual |

---
## 3. Arquitectura Lambda

La **Arquitectura Lambda** fue propuesta por Nathan Marz. Combina procesamiento batch y en tiempo real para ofrecer una vision completa de los datos.

### Diagrama

```
                         +------------------+
                         |   Fuentes de     |
                         |     Datos        |
                         +--------+---------+
                                  |
                     +------------+------------+
                     |                         |
              +------v-------+         +-------v------+
              |  BATCH LAYER |         | SPEED LAYER  |
              |              |         |  (Real-time) |
              | - Almacena   |         | - Procesa    |
              |   todo el    |         |   datos      |
              |   dataset    |         |   recientes  |
              | - Procesa    |         | - Baja       |
              |   en batch   |         |   latencia   |
              | - Alta       |         | - Resultados |
              |   precision  |         |   aprox.     |
              +------+-------+         +-------+------+
                     |                         |
              +------v-------+         +-------v------+
              | Batch Views  |         | Real-time    |
              |              |         | Views        |
              +------+-------+         +-------+------+
                     |                         |
                     +------------+------------+
                                  |
                         +--------v---------+
                         |  SERVING LAYER   |
                         |                  |
                         | Combina batch +  |
                         | real-time views  |
                         | para consultas   |
                         +------------------+
```

### Capas

| Capa | Funcion | Tecnologias tipicas |
|------|---------|--------------------|
| **Batch Layer** | Almacena el dataset completo y genera vistas precalculadas | HDFS, Spark Batch, Hive |
| **Speed Layer** | Procesa datos nuevos en tiempo real para compensar la latencia del batch | Spark Streaming, Storm, Flink |
| **Serving Layer** | Combina resultados de batch y speed para responder consultas | HBase, Cassandra, Druid |

### Ventajas y Desventajas

| Ventajas | Desventajas |
|----------|-------------|
| Tolerante a fallos (el batch corrige errores del speed) | Complejidad: mantener dos pipelines (batch + speed) |
| Resultados precisos (batch) + rapidos (speed) | Duplicacion de logica en dos sistemas diferentes |
| Escalable para grandes volumenes | Mayor costo operacional |
| El batch puede reprocesar todo el historial | Latencia en batch views (horas) |

---
## 4. Arquitectura Kappa

La **Arquitectura Kappa** fue propuesta por Jay Kreps (creador de Kafka). Simplifica Lambda usando **solo streaming** para todo el procesamiento.

### Diagrama

```
     +------------------+
     |   Fuentes de     |
     |     Datos        |
     +--------+---------+
              |
     +--------v---------+
     |  LOG INMUTABLE   |
     |   (ej. Kafka)    |
     | Retiene todo el  |
     | historial de     |
     | eventos          |
     +--------+---------+
              |
     +--------v---------+
     | STREAM PROCESSING|
     |  (ej. Flink,     |
     |   Spark Streaming)|
     |                  |
     | Un solo pipeline |
     | para batch y     |
     | real-time         |
     +--------+---------+
              |
     +--------v---------+
     |  SERVING LAYER   |
     |  Resultados      |
     |  disponibles     |
     |  para consultas  |
     +------------------+
```

### Principio clave
Todo es un **stream de eventos**. Si necesitas reprocesar datos historicos, simplemente re-lees el log desde el inicio.

### Ventajas y Desventajas

| Ventajas | Desventajas |
|----------|-------------|
| Simplicidad: un solo pipeline | Requiere un log inmutable con retencion larga (costo de almacenamiento) |
| Sin duplicacion de logica | Reprocesar grandes volumenes historicos puede ser lento |
| Mas facil de mantener y evolucionar | Menos maduro para algunos escenarios de batch complejo |
| Menor latencia en todos los casos | Depende fuertemente de la tecnologia de streaming |

---
## 5. Comparativa Lambda vs Kappa

| Aspecto | Lambda | Kappa |
|---------|--------|-------|
| **Pipelines** | Dos (batch + speed) | Uno (solo streaming) |
| **Complejidad** | Alta (mantener dos sistemas) | Menor (un solo sistema) |
| **Reprocesamiento** | Batch layer reprocesa todo | Re-leer el log de eventos |
| **Latencia** | Batch: horas / Speed: segundos | Segundos (todo es streaming) |
| **Precision** | Batch corrige errores del speed | Depende de la calidad del streaming |
| **Costo** | Mayor (dos infraestructuras) | Menor infraestructura, mayor almacenamiento de log |
| **Caso ideal** | Necesitas alta precision + tiempo real | Eventos en tiempo real son la fuente de verdad |
| **Ejemplo** | Reportes financieros + dashboard en vivo | Deteccion de fraude, IoT |

---
## 6. On-Premise vs Cloud

| Aspecto | On-Premise | Cloud |
|---------|------------|-------|
| **Costo inicial** | Alto (comprar hardware) | Bajo (pago por uso) |
| **Costo a largo plazo** | Puede ser menor si se usa al maximo | Puede crecer si no se optimiza |
| **Escalabilidad** | Limitada (comprar mas hardware) | Elastica (escalar en minutos) |
| **Mantenimiento** | Equipo propio de infraestructura | El proveedor gestiona el hardware |
| **Seguridad** | Control total | Responsabilidad compartida |
| **Latencia** | Baja (datos locales) | Variable (depende de la region) |
| **Flexibilidad** | Limitada | Alta (servicios managed, serverless) |
| **Time-to-market** | Semanas/meses | Horas/dias |

### Cuando elegir cada opcion?

- **On-Premise:** Regulaciones estrictas de datos (banca, gobierno), carga constante y predecible, ya tienes infraestructura
- **Cloud:** Startups, cargas variables, necesidad de escalar rapidamente, quieres servicios managed
- **Hibrido:** Lo mejor de ambos mundos — datos sensibles on-premise, procesamiento elastico en cloud

---
## 7. Servicios Cloud Equivalentes

Los tres grandes proveedores cloud ofrecen servicios equivalentes para cada componente de una arquitectura Big Data:

| Componente | GCP | AWS | Azure |
|------------|-----|-----|-------|
| **Almacenamiento de objetos** | Cloud Storage | S3 | Blob Storage |
| **Procesamiento distribuido** | Dataproc | EMR | HDInsight |
| **Data Warehouse** | BigQuery | Redshift | Synapse Analytics |
| **Streaming** | Pub/Sub + Dataflow | Kinesis | Event Hubs + Stream Analytics |
| **Orquestacion** | Cloud Composer (Airflow) | Step Functions / MWAA | Data Factory |
| **Base NoSQL** | Bigtable | DynamoDB | Cosmos DB |
| **Catalogo de datos** | Data Catalog | Glue Catalog | Purview |
| **ML Platform** | Vertex AI | SageMaker | Azure ML |
| **Serverless SQL** | BigQuery | Athena | Synapse Serverless |

### Nota sobre costos
Cada proveedor tiene un modelo de precios diferente. Es importante estimar costos antes de elegir, considerando:
- Volumen de datos almacenados
- Cantidad de datos procesados
- Numero de consultas
- Transferencia de datos entre regiones

---
## Ejercicios

Los siguientes ejercicios son **reflexivos**. No requieren codigo, sino analisis y diseno. Escribe tus respuestas en celdas Markdown (cambia el tipo de celda a Markdown) o como comentarios en celdas de codigo.

### EJERCICIO 1: Diseno de arquitectura para e-commerce

**Escenario:** Una empresa de e-commerce tiene:
- Catalogo de 1 millon de productos
- 50.000 ordenes por dia
- Necesidad de recomendaciones de productos en tiempo real
- Reportes diarios de ventas para el equipo de negocio

**TODO:** Disena la arquitectura de datos para este escenario. Responde:
1. Que arquitectura usarias? (Lambda, Kappa, otra)
2. Que componentes/tecnologias elegirais para cada capa?
3. Como manejarias las recomendaciones en tiempo real?
4. Donde almacenarias los datos historicos?
5. Harias on-premise o cloud? Justifica.

Escribe tu respuesta a continuacion (puedes usar esta misma celda o crear nuevas):

In [ ]:
# =============================================================
# EJERCICIO 1: Diseno de arquitectura para e-commerce
# =============================================================
# TODO: Escribe tu diseno de arquitectura como comentarios
# o cambia esta celda a Markdown para escribir con formato.
#
# 1. Arquitectura elegida:
#
# 2. Componentes por capa:
#    - Ingesta:
#    - Procesamiento batch:
#    - Procesamiento real-time:
#    - Almacenamiento:
#    - Serving:
#
# 3. Recomendaciones en tiempo real:
#
# 4. Datos historicos:
#
# 5. On-premise vs Cloud:
#

# Escribe tu respuesta aqui:


### EJERCICIO 2: Arquitectura Lambda para IoT industrial

**Escenario:** Una planta industrial tiene:
- 500 sensores que envian datos cada segundo (temperatura, presion, vibracion)
- Necesidad de detectar anomalias en tiempo real para prevenir fallas
- Reportes mensuales de mantenimiento predictivo
- Almacenamiento de datos historicos por 5 anos (regulacion)

**TODO:** Identifica los componentes de una arquitectura Lambda para este sistema:
1. Que tecnologia usarias para la ingesta de datos de los sensores?
2. Que iria en el Batch Layer? Que procesamiento haria?
3. Que iria en el Speed Layer? Como detectarias anomalias en tiempo real?
4. Que iria en el Serving Layer? Como se consultarian los datos?
5. Que formato de almacenamiento usarias para los datos historicos y por que?

In [ ]:
# =============================================================
# EJERCICIO 2: Arquitectura Lambda para IoT industrial
# =============================================================
# TODO: Identifica los componentes de cada capa de la
# arquitectura Lambda para el sistema de sensores IoT.
#
# 1. Ingesta de datos de sensores:
#
# 2. Batch Layer:
#    - Almacenamiento:
#    - Procesamiento:
#    - Output:
#
# 3. Speed Layer:
#    - Procesamiento:
#    - Deteccion de anomalias:
#    - Alertas:
#
# 4. Serving Layer:
#    - Base de datos:
#    - API/Dashboard:
#
# 5. Formato de almacenamiento historico:
#

# Escribe tu respuesta aqui:


---
## Resumen

En esta actividad aprendimos:

1. **Ecosistema Hadoop:** HDFS (almacenamiento), YARN (recursos), MapReduce (procesamiento) + herramientas como Hive, Pig, Sqoop
2. **Ecosistema moderno:** Spark, Kafka, Airflow y formatos como Parquet y Delta Lake
3. **Arquitectura Lambda:** Tres capas (Batch + Speed + Serving) para combinar precision y velocidad
4. **Arquitectura Kappa:** Un solo pipeline de streaming, simplicidad a costa de cierta flexibilidad
5. **On-Premise vs Cloud:** Trade-offs entre control, costo, escalabilidad y time-to-market
6. **Servicios cloud:** GCP, AWS y Azure ofrecen componentes equivalentes para cada necesidad

---
## Desafio Extra (Opcional)

**Arquitectura para deteccion de fraude bancario en tiempo real**

**Escenario:** Un banco procesa 10 millones de transacciones diarias con tarjeta de credito. Necesita:
- Detectar transacciones fraudulentas en menos de 500ms
- Modelos de ML que se actualizan diariamente con nuevos patrones de fraude
- Dashboard en tiempo real para el equipo de seguridad
- Almacenamiento de todas las transacciones por 7 anos (regulacion bancaria)
- Cumplimiento de regulaciones de privacidad de datos

**TODO:**
1. Elegiras Lambda o Kappa? Justifica tu decision
2. Disena la arquitectura completa con componentes especificos
3. Como integras el modelo de ML en el pipeline de deteccion?
4. Como manejas el reentrenamiento diario del modelo?
5. Que consideraciones de seguridad y privacidad tendrias?

In [ ]:
# =============================================================
# DESAFIO: Arquitectura para deteccion de fraude bancario
# =============================================================
# TODO: Propone una arquitectura completa.
#
# 1. Arquitectura elegida (Lambda/Kappa) y justificacion:
#
# 2. Componentes:
#    - Ingesta de transacciones:
#    - Procesamiento real-time:
#    - Procesamiento batch:
#    - Modelo de ML (serving):
#    - Almacenamiento:
#    - Dashboard:
#
# 3. Integracion del modelo de ML:
#
# 4. Reentrenamiento diario:
#
# 5. Seguridad y privacidad:
#

# Escribe tu respuesta aqui:
